# Week 3: Advanced Python - Solutions

Solutions for functions, generators, regex, and OOP problems.

## Complicated moments explained
These are common points where learners usually get stuck:
- Focus on why each step exists, not only the final answer.
- Compare intermediate outputs to your own attempt, not just final metrics.
- Small implementation differences can still be correct if assumptions match.


## Part 1: Functions

### Problem 1.1: Flexible Sequence Statistics

In [ ]:
def sequence_stats(*sequences, **options):
    """Calculate statistics for multiple sequences.
    
    Args:
        *sequences: Variable number of DNA/RNA sequences
        **options: gc=True, length=True, at_ratio=False
    """
    results = []
    
    show_gc = options.get('gc', True)
    show_length = options.get('length', True)
    show_at_ratio = options.get('at_ratio', False)
    
    for seq in sequences:
        seq = seq.upper()
        stats = {'sequence': seq[:20] + '...' if len(seq) > 20 else seq}
        
        if show_length:
            stats['length'] = len(seq)
        
        if show_gc:
            gc = (seq.count('G') + seq.count('C')) / len(seq) * 100 if seq else 0
            stats['gc_content'] = round(gc, 2)
        
        if show_at_ratio:
            a_count = seq.count('A')
            t_count = seq.count('T') + seq.count('U')  # Handle RNA too
            stats['at_ratio'] = round(a_count / t_count, 3) if t_count > 0 else float('inf')
        
        results.append(stats)
    
    return results

# Test
seqs = ["ATGCGATCGATCG", "GGGGCCCCAAAA", "ATATATAT"]
print(sequence_stats(*seqs, gc=True, at_ratio=True))

### Problem 1.2: Decorator for Timing

In [ ]:
import time
from functools import wraps

def timer(func):
    """Decorator that times function execution."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} took {end - start:.6f} seconds")
        return result
    return wrapper

@timer
def count_kmers(sequence, k):
    """Count k-mers in a sequence."""
    from collections import Counter
    kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    return Counter(kmers)

# Test
result = count_kmers("ATGCATGCATGCATGC" * 1000, 4)
print(f"Found {len(result)} unique 4-mers")

### Problem 1.3: Recursive ORF Finder

In [ ]:
def find_orfs_recursive(sequence, min_length=30, start=0, orfs=None):
    """Find all ORFs recursively."""
    if orfs is None:
        orfs = []
    
    # Base case
    start_pos = sequence.find('ATG', start)
    if start_pos == -1:
        return orfs
    
    # Find stop codon
    stop_codons = ['TAA', 'TAG', 'TGA']
    for i in range(start_pos + 3, len(sequence) - 2, 3):
        codon = sequence[i:i+3]
        if codon in stop_codons:
            orf = sequence[start_pos:i+3]
            if len(orf) >= min_length:
                orfs.append({
                    'start': start_pos,
                    'end': i + 3,
                    'length': len(orf),
                    'sequence': orf
                })
            break
    
    # Recursive call
    return find_orfs_recursive(sequence, min_length, start_pos + 1, orfs)

# Test
test_seq = "ATGAAATTTGGGTAACCCATGCCCGGGTGA"
orfs = find_orfs_recursive(test_seq, min_length=9)
for orf in orfs:
    print(f"ORF at {orf['start']}-{orf['end']}: {orf['sequence']}")

## Part 2: Generators

### Problem 2.1: K-mer Generator

In [ ]:
def kmer_generator(sequence, k):
    """Generate k-mers from a sequence."""
    for i in range(len(sequence) - k + 1):
        yield sequence[i:i+k]

# Test
seq = "ATGCGATCGATCG"
print("4-mers:", list(kmer_generator(seq, 4)))

### Problem 2.2: Sliding Window GC Content Generator

In [ ]:
def gc_windows(sequence, window_size, step=1):
    """Generate GC content for sliding windows."""
    for i in range(0, len(sequence) - window_size + 1, step):
        window = sequence[i:i+window_size]
        gc = (window.count('G') + window.count('C')) / window_size * 100
        yield (i, round(gc, 2))

# Test
seq = "ATGCGCGCATATATATAT"
print("GC content per window:")
for pos, gc in gc_windows(seq, 5, 2):
    print(f"  Position {pos}: {gc}%")

### Problem 2.3: FASTA File Generator

In [ ]:
def parse_fasta_generator(filepath):
    """Parse FASTA file lazily."""
    current_id = None
    current_seq = []
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if current_id is not None:
                    yield (current_id, ''.join(current_seq))
                current_id = line[1:].split()[0]
                current_seq = []
            elif line:
                current_seq.append(line)
        
        if current_id is not None:
            yield (current_id, ''.join(current_seq))

# Usage example (assuming file exists):
# for seq_id, seq in parse_fasta_generator('sequences.fasta'):
#     print(f"{seq_id}: {len(seq)} bp")

### Problem 2.4: Codon Generator with Frame

In [ ]:
def codon_generator(sequence, frame=0):
    """Generate codons from a specific reading frame."""
    for i in range(frame, len(sequence) - 2, 3):
        yield sequence[i:i+3]

def all_frames_generator(sequence):
    """Generate codons from all 6 reading frames."""
    # Forward frames
    for frame in range(3):
        yield (f'+{frame+1}', list(codon_generator(sequence, frame)))
    
    # Reverse complement frames
    complement = str.maketrans('ATGC', 'TACG')
    rev_comp = sequence.translate(complement)[::-1]
    for frame in range(3):
        yield (f'-{frame+1}', list(codon_generator(rev_comp, frame)))

# Test
seq = "ATGAAACCCGGG"
for frame_name, codons in all_frames_generator(seq):
    print(f"Frame {frame_name}: {codons}")

## Part 3: Regular Expressions

In [ ]:
import re

### Problem 3.1: Find Restriction Sites

In [ ]:
RESTRICTION_ENZYMES = {
    'EcoRI': 'GAATTC',
    'BamHI': 'GGATCC',
    'HindIII': 'AAGCTT',
    'NotI': 'GCGGCCGC',
    'XbaI': 'TCTAGA',
    'SalI': 'GTCGAC',
    'PstI': 'CTGCAG'
}

def find_restriction_sites(sequence, enzymes=None):
    """Find restriction enzyme cut sites in a sequence."""
    if enzymes is None:
        enzymes = RESTRICTION_ENZYMES
    
    results = {}
    for enzyme, site in enzymes.items():
        positions = [m.start() for m in re.finditer(site, sequence)]
        if positions:
            results[enzyme] = positions
    
    return results

# Test
seq = "ATGAATTCGGATCCAAGCTTGAATTCTCTAGA"
sites = find_restriction_sites(seq)
for enzyme, positions in sites.items():
    print(f"{enzyme}: positions {positions}")

### Problem 3.2: Extract Protein Motifs

In [ ]:
PROTEIN_MOTIFS = {
    'N-glycosylation': r'N[^P][ST][^P]',  # N-X-S/T-X where X is not P
    'Nuclear_localization': r'[KR]{4,}',   # 4+ consecutive K or R
    'Zinc_finger_C2H2': r'C.{2,4}C.{12}H.{3,5}H',
    'ATP_binding': r'[AG].{4}GK[ST]'
}

def find_protein_motifs(protein_seq, motifs=None):
    """Find protein motifs in a sequence."""
    if motifs is None:
        motifs = PROTEIN_MOTIFS
    
    results = {}
    for motif_name, pattern in motifs.items():
        matches = []
        for m in re.finditer(pattern, protein_seq):
            matches.append({
                'position': m.start(),
                'match': m.group()
            })
        if matches:
            results[motif_name] = matches
    
    return results

# Test
protein = "MNGTSVKRRRRLSAAAGGGKSTDDD"
motifs = find_protein_motifs(protein)
for name, matches in motifs.items():
    print(f"{name}:")
    for m in matches:
        print(f"  Position {m['position']}: {m['match']}")

### Problem 3.3: Parse GenBank Features

In [ ]:
def parse_genbank_location(location_str):
    """Parse GenBank location string."""
    # Handle complement
    complement = False
    if location_str.startswith('complement('):
        complement = True
        location_str = location_str[11:-1]
    
    # Handle join
    if location_str.startswith('join('):
        location_str = location_str[5:-1]
        ranges = location_str.split(',')
    else:
        ranges = [location_str]
    
    # Parse ranges
    positions = []
    for r in ranges:
        r = r.strip()
        if '..' in r:
            parts = r.replace('<', '').replace('>', '').split('..')
            positions.append((int(parts[0]), int(parts[1])))
        else:
            pos = int(r.replace('<', '').replace('>', ''))
            positions.append((pos, pos))
    
    return {'complement': complement, 'positions': positions}

# Test
test_locations = [
    "100..500",
    "complement(200..800)",
    "join(100..200,300..400,500..600)",
    "complement(join(100..200,300..400))"
]

for loc in test_locations:
    print(f"{loc} -> {parse_genbank_location(loc)}")

## Part 4: Object-Oriented Programming

### Problem 4.1: DNASequence Class

In [ ]:
class DNASequence:
    """Class representing a DNA sequence."""
    
    COMPLEMENT = str.maketrans('ATGC', 'TACG')
    
    def __init__(self, sequence, name="unnamed"):
        self.sequence = sequence.upper()
        self.name = name
        self._validate()
    
    def _validate(self):
        """Validate that sequence contains only valid nucleotides."""
        valid = set('ATGCN')
        if not all(nuc in valid for nuc in self.sequence):
            raise ValueError("Invalid nucleotides in sequence")
    
    def __len__(self):
        return len(self.sequence)
    
    def __str__(self):
        return f">{self.name}\n{self.sequence}"
    
    def __repr__(self):
        return f"DNASequence('{self.sequence[:20]}...', name='{self.name}')"
    
    def __getitem__(self, key):
        return DNASequence(self.sequence[key], f"{self.name}_slice")
    
    def __add__(self, other):
        if isinstance(other, DNASequence):
            return DNASequence(self.sequence + other.sequence, 
                             f"{self.name}+{other.name}")
        return NotImplemented
    
    @property
    def gc_content(self):
        """Calculate GC content as percentage."""
        gc = self.sequence.count('G') + self.sequence.count('C')
        return round(gc / len(self.sequence) * 100, 2)
    
    def complement(self):
        """Return the complement sequence."""
        return DNASequence(self.sequence.translate(self.COMPLEMENT), 
                          f"{self.name}_complement")
    
    def reverse_complement(self):
        """Return the reverse complement."""
        return DNASequence(self.sequence.translate(self.COMPLEMENT)[::-1],
                          f"{self.name}_revcomp")
    
    def transcribe(self):
        """Transcribe DNA to RNA."""
        return self.sequence.replace('T', 'U')
    
    def find_orfs(self, min_length=30):
        """Find all ORFs in the sequence."""
        orfs = []
        stop_codons = {'TAA', 'TAG', 'TGA'}
        
        for frame in range(3):
            i = frame
            while i < len(self.sequence) - 2:
                if self.sequence[i:i+3] == 'ATG':
                    for j in range(i + 3, len(self.sequence) - 2, 3):
                        if self.sequence[j:j+3] in stop_codons:
                            if j + 3 - i >= min_length:
                                orfs.append({
                                    'start': i,
                                    'end': j + 3,
                                    'frame': frame + 1,
                                    'sequence': self.sequence[i:j+3]
                                })
                            break
                i += 3
        
        return orfs

# Test
dna = DNASequence("ATGAAACCCGGGTAAGGG", "test_gene")
print(f"Length: {len(dna)}")
print(f"GC content: {dna.gc_content}%")
print(f"Reverse complement: {dna.reverse_complement().sequence}")
print(f"ORFs: {dna.find_orfs(min_length=9)}")

### Problem 4.2: Sequence Collection with Inheritance

In [ ]:
from abc import ABC, abstractmethod

class Sequence(ABC):
    """Abstract base class for biological sequences."""
    
    def __init__(self, sequence, name="unnamed"):
        self.sequence = sequence.upper()
        self.name = name
        self._validate()
    
    @property
    @abstractmethod
    def alphabet(self):
        """Return valid characters for this sequence type."""
        pass
    
    def _validate(self):
        invalid = set(self.sequence) - self.alphabet
        if invalid:
            raise ValueError(f"Invalid characters: {invalid}")
    
    def __len__(self):
        return len(self.sequence)
    
    @abstractmethod
    def molecular_weight(self):
        """Calculate molecular weight."""
        pass


class DNA(Sequence):
    """DNA sequence class."""
    
    @property
    def alphabet(self):
        return set('ATGCN')
    
    def molecular_weight(self):
        weights = {'A': 331.2, 'T': 322.2, 'G': 347.2, 'C': 307.2, 'N': 327.0}
        return sum(weights.get(n, 0) for n in self.sequence)


class RNA(Sequence):
    """RNA sequence class."""
    
    @property
    def alphabet(self):
        return set('AUGCN')
    
    def molecular_weight(self):
        weights = {'A': 347.2, 'U': 324.2, 'G': 363.2, 'C': 323.2, 'N': 340.0}
        return sum(weights.get(n, 0) for n in self.sequence)


class Protein(Sequence):
    """Protein sequence class."""
    
    AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY*'
    
    @property
    def alphabet(self):
        return set(self.AMINO_ACIDS + 'X')
    
    def molecular_weight(self):
        weights = {
            'A': 89.1, 'C': 121.2, 'D': 133.1, 'E': 147.1, 'F': 165.2,
            'G': 75.1, 'H': 155.2, 'I': 131.2, 'K': 146.2, 'L': 131.2,
            'M': 149.2, 'N': 132.1, 'P': 115.1, 'Q': 146.2, 'R': 174.2,
            'S': 105.1, 'T': 119.1, 'V': 117.1, 'W': 204.2, 'Y': 181.2,
            '*': 0, 'X': 110.0
        }
        # Subtract water for peptide bonds
        total = sum(weights.get(aa, 0) for aa in self.sequence)
        return total - 18.0 * (len(self.sequence) - 1)


# Test
dna = DNA("ATGCATGC", "my_dna")
rna = RNA("AUGCAUGC", "my_rna")
protein = Protein("MKTL", "my_protein")

print(f"DNA MW: {dna.molecular_weight():.1f} Da")
print(f"RNA MW: {rna.molecular_weight():.1f} Da")
print(f"Protein MW: {protein.molecular_weight():.1f} Da")

### Problem 4.3: FASTA Collection Manager

In [ ]:
class FASTACollection:
    """Manager for collections of FASTA sequences."""
    
    def __init__(self):
        self._sequences = {}
    
    def add(self, seq_id, sequence):
        """Add a sequence to the collection."""
        self._sequences[seq_id] = sequence
    
    def remove(self, seq_id):
        """Remove a sequence by ID."""
        if seq_id in self._sequences:
            del self._sequences[seq_id]
    
    def __getitem__(self, seq_id):
        return self._sequences[seq_id]
    
    def __len__(self):
        return len(self._sequences)
    
    def __iter__(self):
        return iter(self._sequences.items())
    
    def __contains__(self, seq_id):
        return seq_id in self._sequences
    
    @classmethod
    def from_file(cls, filepath):
        """Load collection from FASTA file."""
        collection = cls()
        current_id = None
        current_seq = []
        
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('>'):
                    if current_id:
                        collection.add(current_id, ''.join(current_seq))
                    current_id = line[1:].split()[0]
                    current_seq = []
                elif line:
                    current_seq.append(line)
            
            if current_id:
                collection.add(current_id, ''.join(current_seq))
        
        return collection
    
    def to_file(self, filepath, line_width=60):
        """Save collection to FASTA file."""
        with open(filepath, 'w') as f:
            for seq_id, seq in self:
                f.write(f">{seq_id}\n")
                for i in range(0, len(seq), line_width):
                    f.write(seq[i:i+line_width] + '\n')
    
    def filter_by_length(self, min_len=0, max_len=float('inf')):
        """Return new collection with sequences in length range."""
        filtered = FASTACollection()
        for seq_id, seq in self:
            if min_len <= len(seq) <= max_len:
                filtered.add(seq_id, seq)
        return filtered
    
    def filter_by_gc(self, min_gc=0, max_gc=100):
        """Return new collection with sequences in GC content range."""
        filtered = FASTACollection()
        for seq_id, seq in self:
            gc = (seq.count('G') + seq.count('C')) / len(seq) * 100
            if min_gc <= gc <= max_gc:
                filtered.add(seq_id, seq)
        return filtered
    
    def summary(self):
        """Return summary statistics."""
        lengths = [len(seq) for _, seq in self]
        if not lengths:
            return {"count": 0}
        
        return {
            "count": len(lengths),
            "total_bp": sum(lengths),
            "min_length": min(lengths),
            "max_length": max(lengths),
            "avg_length": sum(lengths) / len(lengths)
        }

# Test
collection = FASTACollection()
collection.add("seq1", "ATGCATGCATGC")
collection.add("seq2", "GGGGCCCCGGGGCCCC")
collection.add("seq3", "AAAAAATTTTTT")

print(f"Collection has {len(collection)} sequences")
print(f"Summary: {collection.summary()}")

high_gc = collection.filter_by_gc(min_gc=50)
print(f"High GC sequences: {[s[0] for s in high_gc]}")

---

## Summary

Key concepts demonstrated:

1. **Functions**: `*args`, `**kwargs`, decorators, recursion
2. **Generators**: `yield`, lazy evaluation, memory efficiency
3. **Regex**: Pattern matching, capturing groups, lookahead/lookbehind
4. **OOP**: Classes, inheritance, `@property`, magic methods, abstract classes